# Adaptive A\* — independent re-implementation and reproduction (Kaggle runner)

Re-implementation of H. Cao *et al.*, "An Adaptive A\* Algorithm for Mobile Robots Global Path Planning," *Electronics* 15(13), 2807, 2026.

This notebook runs the whole pipeline end to end:
1. unit tests;
2. reproduction of Tables 1–5 against the published numbers, plus the 50-map robustness study;
3. the paired timing study;
4. all figures.

It then shows the results inline and packs them into a zip you can download.

**CPU only.** No GPU or accelerator is needed. A full run takes about 5–10 minutes.

### How to get the code into the notebook (choose one)
- **A. Kaggle Dataset (works offline).** Upload the `adaptive-astar-reimplementation` folder (or a zip of it) as a Kaggle Dataset, then *Add Input* → that dataset. The next cell finds it automatically under `/kaggle/input`.
- **B. GitHub clone (needs Internet).** Turn on *Settings → Internet*, then set `REPO_URL` / `BRANCH` below. A private repo needs a personal access token: add it under *Add-ons → Secrets* as `GITHUB_TOKEN`.

In [ ]:
# ---- configuration -------------------------------------------------------
REPO_URL = "https://github.com/Mariam1543/aqqal123.git"   # used only for option B
BRANCH = "claude/relaxed-johnson-4f4051"

QUICK = False          # True = smoke run (about 1 min); False = full run as in the report
TIMING_REPEATS = 3 if QUICK else 30
ROBUST_MAPS = 5 if QUICK else 50
TIMING_PAIRS = 30 if QUICK else 300

In [ ]:
# ---- locate the code and copy it to a writable directory -------------------
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
PROJECT = WORK / "adaptive-astar-reimplementation"

def find_in_inputs():
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for marker in root.rglob("adaptive_astar/__init__.py"):          # unpacked folder
        return marker.parents[2]
    for z in root.rglob("*.zip"):                                     # zipped upload
        with zipfile.ZipFile(z) as zf:
            zf.extractall(WORK / "_unzipped")
        for marker in (WORK / "_unzipped").rglob("adaptive_astar/__init__.py"):
            return marker.parents[2]
    return None

if not (PROJECT / "src" / "adaptive_astar").exists():
    src = find_in_inputs()
    if src is not None:                                               # option A
        shutil.copytree(src, PROJECT, dirs_exist_ok=True)
        print("copied code from Kaggle input:", src)
    else:                                                             # option B
        url = REPO_URL
        token = None
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass
        if token:
            url = url.replace("https://", f"https://{token}@")
        clone_dir = WORK / "_repo"
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(clone_dir)], check=True)
        shutil.copytree(clone_dir / "adaptive-astar-reimplementation", PROJECT, dirs_exist_ok=True)
        print("cloned", REPO_URL, "@", BRANCH)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
print("project directory:", PROJECT)
print(sorted(p.name for p in PROJECT.iterdir()))

In [ ]:
# ---- environment check (all packages are pre-installed on Kaggle) --------
import platform, numpy, pandas, scipy, matplotlib
print("python", platform.python_version(), "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)
print("CPUs:", os.cpu_count())

In [ ]:
def run(*args):
    """Run a project script, streaming its output; stop the notebook on failure."""
    print("$ python", " ".join(args), flush=True)
    proc = subprocess.run([sys.executable, *args], cwd=PROJECT, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-6000:])
    if proc.returncode != 0:
        raise RuntimeError(f"{args[0]} failed with exit code {proc.returncode}")

run("-m", "unittest", "discover", "-s", "tests")

## Reproduce Tables 1–5 and the robustness study

This uses the calibrated proxy maps already committed in `maps/`. Do **not** re-run `archive/calibrate_maps.py` with a smaller budget: it would overwrite the maps and change every result. The default command, `python archive/calibrate_maps.py --budget-scale 4`, rebuilds exactly the same maps.

In [ ]:
run("scripts/run_experiments.py", "--repeats", str(TIMING_REPEATS), "--robust-maps", str(ROBUST_MAPS))

In [ ]:
run("scripts/timing_study.py", "--pairs", str(TIMING_PAIRS))
run("scripts/make_figures.py")

## Results

In [ ]:
from IPython.display import Markdown, Image, display

display(Markdown((PROJECT / "results" / "timing_study.md").read_text()))
comparison = (PROJECT / "results" / "comparison.md").read_text()
display(Markdown(comparison.split("# Published vs reproduced results")[0]))   # headline table

In [ ]:
# Full paper-vs-ours tables (Tables 1-5) and the robustness study
display(Markdown(comparison))
display(Markdown((PROJECT / "results" / "robustness.md").read_text()))

In [ ]:
for name in ["fig_relative_change", "fig_robustness", "fig_weight_heuristic",
             "fig_ablation_30x30", "fig_ablation_60x60",
             "fig_algorithms_30x30", "fig_algorithms_60x60", "fig_algorithms_100x100",
             "fig_start_goal"]:
    print(name)
    display(Image(filename=str(PROJECT / "figures" / f"{name}.png")))

In [ ]:
# ---- package outputs for download (Output tab -> adaptive_astar_outputs.zip) --
out = WORK / "adaptive_astar_outputs.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("results", "figures", "maps"):
        for f in sorted((PROJECT / folder).rglob("*")):
            if f.is_file():
                zf.write(f, f.relative_to(PROJECT))
print("download:", out, f"({out.stat().st_size / 1e6:.1f} MB)")
